# 45x35.5 테이블 crop + 평면화, 배경색 성능 비교

`background/` 폴더에는 흰색 책상과 검정 매트, 두 가지 배경 위에 가로 45cm x 세로 35.5cm
영역의 네 모서리에 초록색 테이프를 붙이고 같은 타일 배치를 촬영한 사진이 섞여 있습니다.
`test5.ipynb`(→ `test7.ipynb`)의 초록 테이프 검출/평면화 로직을 그대로 재사용하되, 이번에는
전처리를 두 단계로 나눕니다.

1. **Crop**: 검출된 4개 테이프 좌표를 감싸는 축 정렬 사각형(bounding box)으로 원본을 자릅니다
   (원근 보정 없이 잘라내기만 함).
2. **평면화**: 1번에서 자른 이미지를 대상으로, 좌표를 crop 기준으로 다시 맞춘 뒤
   45:35.5cm 비율로 `cv2.getPerspectiveTransform` + `cv2.warpPerspective` 워핑합니다.

두 결과 이미지(`_crop.jpg`, `_flat.jpg`)를 모두 `background_result/`에 저장하고,
`best_v8.pt`로 원본/crop/평면화 3단계 x 배경색(밝음/어두움) 조합별 탐지 성능을 비교합니다.

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "background"
OUTPUT_DIR = BASE_DIR / "background_result"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

# 초록색 테이프를 검출하기 위한 HSV 범위 (필요 시 조정)
GREEN_HSV_LOWER = np.array([35, 60, 40])
GREEN_HSV_UPPER = np.array([85, 255, 255])

# 컨투어 필터링: 전체 이미지 면적 대비 테이프 영역의 최소/최대 비율
MIN_AREA_RATIO = 0.0003
MAX_AREA_RATIO = 0.05

# 테이블 실제 크기(cm)와 평면화 출력 해상도(px) 설정
TABLE_SHORT_CM = 35.5
TABLE_LONG_CM = 45.0
LONG_SIDE_PX = 1200
SHORT_SIDE_PX = round(LONG_SIDE_PX * TABLE_SHORT_CM / TABLE_LONG_CM)

# 배경색 자동 분류 기준: crop 이미지의 평균 명도(HSV V채널)가 이 값 이상이면 밝은 배경
BRIGHTNESS_THRESHOLD = 128

if not DATA_DIR.exists():
    raise FileNotFoundError(f"이미지 폴더가 없습니다: {DATA_DIR.resolve()}")

OUTPUT_DIR.mkdir(exist_ok=True)

image_paths = sorted(
    p for p in DATA_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)
print(f"대상 이미지: {len(image_paths)}장")
for p in image_paths:
    print(" -", p.name)

In [ ]:
import matplotlib.font_manager as fm

# matplotlib 기본 폰트는 한글을 지원하지 않아 그래프 제목이 네모(□)로 깨져 보입니다.
# 1) 시스템에 이미 설치된 한글 폰트를 먼저 찾고,
# 2) 없으면 노트북 옆의 `fonts/` 폴더에 놓아둔 .ttf 파일을 찾아 등록합니다.
#    (인터넷이 안 되는 폐쇄망 서버라면, 로컬 PC의 C:\Windows\Fonts\malgun.ttf 파일을
#     이 프로젝트의 `fonts/` 폴더로 복사해 넣고 이 셀을 다시 실행하세요.)
_installed_fonts = {f.name for f in fm.fontManager.ttflist}
_font_name = next(
    (n for n in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR", "Noto Sans KR"]
     if n in _installed_fonts),
    None,
)

if _font_name is None:
    _font_dir = BASE_DIR / "fonts"
    _local_fonts = sorted(_font_dir.glob("*.ttf")) if _font_dir.exists() else []
    if _local_fonts:
        fm.fontManager.addfont(str(_local_fonts[0]))
        _font_name = fm.FontProperties(fname=str(_local_fonts[0])).get_name()
        print(f"로컬 폰트 등록: {_local_fonts[0].name} -> {_font_name}")
    else:
        print(
            "한글 폰트를 찾지 못했습니다. 그래프 제목이 깨질 수 있습니다.\n"
            f"해결하려면 한글 .ttf 폰트 파일을 '{_font_dir}' 폴더에 넣고 이 셀을 다시 실행하세요."
        )

if _font_name:
    plt.rcParams["font.family"] = _font_name

plt.rcParams["axes.unicode_minus"] = False  # 한글 폰트 사용 시 마이너스(-) 기호 깨짐 방지

In [ ]:
def order_points(pts: np.ndarray) -> np.ndarray:
    """4개의 점을 좌상, 우상, 우하, 좌하 순서로 정렬합니다.

    단순 x+y/x-y 기준 정렬은 테이블이 큰 각도로 회전/기울어진 경우
    두 꼭짓점이 같은 점으로 잘못 뽑히는 경우가 있어, 중심점 기준 각도로
    정렬하는 방식(회전에 안정적)을 사용합니다.
    """
    center = pts.mean(axis=0)
    angles = np.arctan2(pts[:, 1] - center[1], pts[:, 0] - center[0])
    ordered = pts[np.argsort(angles)]

    # 정렬된 4점 중 좌상(x+y 최소)에 해당하는 점부터 시작하도록 회전
    start = np.argmin(ordered.sum(axis=1))
    ordered = np.roll(ordered, -start, axis=0)

    return ordered.astype(np.float32)


def detect_green_corners(image: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """이미지에서 초록색 테이프 4개의 중심점을 찾아 정렬된 좌표와 마스크를 반환합니다."""
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, GREEN_HSV_LOWER, GREEN_HSV_UPPER)

    kernel = np.ones((7, 7), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    image_area = image.shape[0] * image.shape[1]
    min_area = image_area * MIN_AREA_RATIO
    max_area = image_area * MAX_AREA_RATIO

    candidates = [c for c in contours if min_area <= cv2.contourArea(c) <= max_area]
    candidates = sorted(candidates, key=cv2.contourArea, reverse=True)[:4]

    if len(candidates) != 4:
        raise ValueError(
            f"초록색 테이프를 4개 찾지 못했습니다 (검출: {len(candidates)}개). "
            "GREEN_HSV_LOWER/UPPER 또는 면적 비율을 조정하세요."
        )

    centers = []
    for c in candidates:
        M = cv2.moments(c)
        cx = M["m10"] / M["m00"]
        cy = M["m01"] / M["m00"]
        centers.append([cx, cy])

    corners = order_points(np.array(centers, dtype=np.float32))
    return corners, mask

In [ ]:
def bounding_box(corners: np.ndarray, shape: tuple[int, int]) -> tuple[int, int, int, int]:
    """corners를 감싸는 축 정렬 사각형을 이미지 범위 안으로 클리핑해 반환합니다."""
    height, width = shape
    x_min, y_min = corners.min(axis=0)
    x_max, y_max = corners.max(axis=0)
    x1 = max(int(np.floor(x_min)), 0)
    y1 = max(int(np.floor(y_min)), 0)
    x2 = min(int(np.ceil(x_max)), width)
    y2 = min(int(np.ceil(y_max)), height)
    return x1, y1, x2, y2


def crop_to_corners(image: np.ndarray, corners: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """테이프 4점을 감싸는 사각형으로 원본을 자르고, 좌표를 crop 기준으로 다시 맞춥니다."""
    x1, y1, x2, y2 = bounding_box(corners, image.shape[:2])
    cropped = image[y1:y2, x1:x2].copy()
    shifted_corners = corners - np.array([x1, y1], dtype=np.float32)
    return cropped, shifted_corners


def warp_to_table(image: np.ndarray, corners: np.ndarray) -> np.ndarray:
    """정렬된 4개 꼭짓점(tl, tr, br, bl)을 45x35.5 테이블 비율의 탑다운 뷰로 워핑합니다."""
    tl, tr, br, bl = corners

    width_top = np.linalg.norm(tr - tl)
    width_bottom = np.linalg.norm(br - bl)
    height_left = np.linalg.norm(bl - tl)
    height_right = np.linalg.norm(br - tr)

    measured_width = max(width_top, width_bottom)
    measured_height = max(height_left, height_right)

    # 측정된 가로/세로 비율이 테이블의 어느 방향(가로/세로)에 더 가까운지로 출력 방향 결정
    if measured_width >= measured_height:
        out_w, out_h = LONG_SIDE_PX, SHORT_SIDE_PX  # 가로(landscape, 45cm 변이 가로)
    else:
        out_w, out_h = SHORT_SIDE_PX, LONG_SIDE_PX  # 세로(portrait, 45cm 변이 세로)

    dst = np.array(
        [
            [0, 0],
            [out_w - 1, 0],
            [out_w - 1, out_h - 1],
            [0, out_h - 1],
        ],
        dtype=np.float32,
    )

    matrix = cv2.getPerspectiveTransform(corners, dst)
    warped = cv2.warpPerspective(image, matrix, (out_w, out_h))
    return warped


def classify_background(image: np.ndarray) -> tuple[str, float]:
    """crop 이미지의 평균 명도(HSV V채널)로 밝은/어두운 배경을 자동 분류합니다."""
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    v_mean = float(hsv[:, :, 2].mean())
    label = "밝은 배경(화이트 테이블)" if v_mean >= BRIGHTNESS_THRESHOLD else "어두운 배경(블랙 매트)"
    return label, v_mean


def process_image(path: Path):
    image = cv2.imread(str(path))
    if image is None:
        raise FileNotFoundError(f"이미지를 읽을 수 없습니다: {path}")

    corners, mask = detect_green_corners(image)
    cropped, shifted_corners = crop_to_corners(image, corners)
    warped = warp_to_table(cropped, shifted_corners)

    annotated = image.copy()
    labels = ["TL", "TR", "BR", "BL"]
    for (x, y), label in zip(corners, labels):
        cv2.circle(annotated, (int(x), int(y)), 12, (0, 0, 255), 3)
        cv2.putText(
            annotated, label, (int(x) + 15, int(y)),
            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2, cv2.LINE_AA,
        )
    cv2.polylines(annotated, [corners.astype(np.int32)], True, (0, 255, 255), 2)
    x1, y1, x2, y2 = bounding_box(corners, image.shape[:2])
    cv2.rectangle(annotated, (x1, y1), (x2, y2), (255, 0, 0), 3)

    return annotated, mask, cropped, warped

In [ ]:
records = []

for image_path in image_paths:
    try:
        annotated, mask, cropped, warped = process_image(image_path)
    except ValueError as e:
        print(f"[{image_path.name}] 실패: {e}")
        continue

    crop_path = OUTPUT_DIR / f"{image_path.stem}_crop.jpg"
    flat_path = OUTPUT_DIR / f"{image_path.stem}_flat.jpg"
    cv2.imwrite(str(crop_path), cropped)
    cv2.imwrite(str(flat_path), warped)

    background_label, brightness = classify_background(cropped)
    records.append({
        "stem": image_path.stem,
        "raw_path": image_path,
        "crop_path": crop_path,
        "flat_path": flat_path,
        "background": background_label,
        "brightness": brightness,
    })
    print(f"[{image_path.name}] {background_label} (명도 {brightness:.1f}) -> {crop_path.name}, {flat_path.name}")

    fig, axes = plt.subplots(1, 4, figsize=(20, 6))

    axes[0].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"{image_path.name}\n원본 + 검출된 모서리")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("초록색 테이프 마스크")
    axes[1].axis("off")

    axes[2].imshow(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))
    axes[2].set_title("1. Crop 결과")
    axes[2].axis("off")

    axes[3].imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
    axes[3].set_title(f"2. 평면화 결과 (45x35.5)\n{background_label}")
    axes[3].axis("off")

    plt.tight_layout()
    plt.show()

print(f"\n총 {len(records)}장 처리 완료")

## 튜닝 가이드

- 테이프가 검출되지 않거나(`실패` 출력) 잘못된 영역이 잡히면 `GREEN_HSV_LOWER`/`GREEN_HSV_UPPER` 값을 조정하세요.
  가운데 마스크 이미지(`초록색 테이프 마스크`)를 보면서 테이프 4개만 하얗게 나오도록 맞추면 됩니다.
- 테이프 외에 다른 초록색 물체가 같이 잡히면 `MIN_AREA_RATIO` / `MAX_AREA_RATIO`로 면적 범위를 좁히세요.
- 평면화 출력 해상도를 바꾸고 싶으면 `LONG_SIDE_PX` 값만 조정하면 `SHORT_SIDE_PX`가 테이블 비율(45:35.5)에
  맞춰 자동 계산됩니다.
- 테이블 실측 크기가 다르면 `TABLE_SHORT_CM` / `TABLE_LONG_CM` 값만 바꾸면 됩니다.
- 배경색 자동 분류가 실제와 다르게 나오면(예: 특히 어두운 흰 책상이나 밝은 매트) `BRIGHTNESS_THRESHOLD`를 조정하세요.
- 전처리 결과는 `background_result/`에 `{원본파일명}_crop.jpg`(1단계), `{원본파일명}_flat.jpg`(2단계)로 저장됩니다.

## 원본 vs Crop vs 평면화 x 배경색 YOLO 탐지 비교

(`test4.ipynb`/`test7.ipynb` 참고) 같은 `best_v8.pt` 모델로 원본/Crop/평면화 3단계 이미지를 각각
탐지하고, 위에서 자동 분류한 배경색(밝은 화이트 테이블 vs 어두운 블랙 매트)별로 묶어서
전처리 단계와 배경색이 탐지 성능에 미치는 영향을 함께 비교합니다.

In [ ]:
import torch
from ultralytics import YOLO

MODEL_PATH = BASE_DIR / "best_v8.pt"
CONFIDENCE = 0.25

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"모델 파일이 없습니다: {MODEL_PATH.resolve()}")
if not records:
    raise RuntimeError("처리된 이미지가 없습니다. 위쪽 전처리 셀을 먼저 실행하세요.")

device = 0 if torch.cuda.is_available() else "cpu"
print("사용 장치:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model = YOLO(str(MODEL_PATH))

In [ ]:
def _count_detections(result) -> int:
    boxes = result.boxes
    return len(boxes) if boxes is not None else 0


summary = []

for record in records:
    raw_result = model.predict(source=str(record["raw_path"]), conf=CONFIDENCE, device=device, verbose=False)[0]
    crop_result = model.predict(source=str(record["crop_path"]), conf=CONFIDENCE, device=device, verbose=False)[0]
    flat_result = model.predict(source=str(record["flat_path"]), conf=CONFIDENCE, device=device, verbose=False)[0]

    raw_count = _count_detections(raw_result)
    crop_count = _count_detections(crop_result)
    flat_count = _count_detections(flat_result)

    fig, axes = plt.subplots(1, 3, figsize=(20, 7))

    axes[0].imshow(cv2.cvtColor(raw_result.plot(), cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"원본\n{record['stem']} ({raw_count}개)")
    axes[0].axis("off")

    axes[1].imshow(cv2.cvtColor(crop_result.plot(), cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Crop\n{record['stem']} ({crop_count}개)")
    axes[1].axis("off")

    axes[2].imshow(cv2.cvtColor(flat_result.plot(), cv2.COLOR_BGR2RGB))
    axes[2].set_title(f"평면화\n{record['stem']} ({flat_count}개)")
    axes[2].axis("off")

    fig.suptitle(record["background"])
    plt.tight_layout()
    plt.show()

    summary.append({
        "stem": record["stem"],
        "background": record["background"],
        "raw_count": raw_count,
        "crop_count": crop_count,
        "flat_count": flat_count,
    })

In [ ]:
print(f"{'파일':38s} {'배경':22s} {'원본':>6s} {'Crop':>6s} {'평탄화':>6s}")
for row in summary:
    print(
        f"{row['stem']:38s} {row['background']:22s} "
        f"{row['raw_count']:>6d} {row['crop_count']:>6d} {row['flat_count']:>6d}"
    )

In [ ]:
from collections import defaultdict

stage_keys = [("원본", "raw_count"), ("Crop", "crop_count"), ("평탄화", "flat_count")]
groups = defaultdict(lambda: {label: [] for label, _ in stage_keys})

for row in summary:
    for label, key in stage_keys:
        groups[row["background"]][label].append(row[key])

print(f"{'배경':22s} {'원본 평균':>10s} {'Crop 평균':>10s} {'평탄화 평균':>10s}")
for background, counts in groups.items():
    averages = {label: sum(vals) / len(vals) for label, vals in counts.items()}
    print(
        f"{background:22s} "
        f"{averages['원본']:>10.1f} {averages['Crop']:>10.1f} {averages['평탄화']:>10.1f}"
    )

stage_labels = [label for label, _ in stage_keys]
backgrounds = list(groups.keys())
x = np.arange(len(stage_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
for i, background in enumerate(backgrounds):
    offset = (i - (len(backgrounds) - 1) / 2) * width
    averages = [sum(groups[background][label]) / len(groups[background][label]) for label in stage_labels]
    ax.bar(x + offset, averages, width, label=background)

ax.set_xticks(x)
ax.set_xticklabels(stage_labels)
ax.set_ylabel("평균 탐지 개수")
ax.set_title("배경색 x 전처리 단계별 평균 탐지 개수")
ax.legend()
plt.tight_layout()
plt.show()